# Notebook 63 — Blockade Boundary and Universality Transition

This notebook extends Notebook 62.

The goal is to connect the universality exponent `b` to the onset of the
Rydberg blockade regime.

We track how the fitted stretched-exponential exponent changes as the
interaction strength crosses the simplified blockade boundary

`V / Ω ≈ 1`

and test whether the universality exponent detects this transition.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


In [ ]:
x = np.linspace(1e-3, 1.0, 500)

def build_S_from_gamma(gamma_x, x):
    dx = x[1] - x[0]
    integral = np.cumsum(gamma_x) * dx
    return np.exp(-integral), integral

def stretched(x, a, b):
    return np.exp(-a * np.power(x, b))

def fit_stretched(x, S):
    popt, _ = curve_fit(
        stretched,
        np.clip(x, 1e-12, None),
        np.clip(S, 1e-12, 1.0),
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 3.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return a, b, pred, mse

def gamma_eff(x, Omega, Delta, V, gamma, gamma_phi):
    base = 1.5 + 0.5 * (Omega / (Omega + abs(Delta) + 1e-8))
    deph = gamma_phi * np.exp(-((x - 0.3) / 0.2)**2)
    interaction = V * np.sin(2 * np.pi * x)
    decay = gamma
    return np.clip(base + deph + interaction + decay, 0.2, None)

def b_local_from_gamma(x, gamma_x, S):
    integral = -np.log(np.clip(S, 1e-12, None))
    return x * gamma_x / np.clip(integral, 1e-12, None)


## Sweep blockade ratio V / Ω

In [ ]:
Omega = 1.0
Delta = 0.2
gamma = 0.3
gamma_phi = 0.5

V_vals = np.linspace(0.0, 2.0, 41)
ratio_vals = V_vals / Omega

b_vals = []
mse_vals = []
S_examples = {}
gamma_examples = {}
b_local_examples = {}

selected_targets = [0.5, 1.0, 1.5]

for V in V_vals:
    gamma_x = gamma_eff(x, Omega, Delta, V, gamma, gamma_phi)
    S, I = build_S_from_gamma(gamma_x, x)
    a_fit, b_fit, S_fit, mse = fit_stretched(x, S)
    b_vals.append(b_fit)
    mse_vals.append(mse)

    if np.any(np.isclose(V / Omega, selected_targets, atol=0.03)):
        key = f"V/Omega={V/Omega:.2f}"
        S_examples[key] = S
        gamma_examples[key] = gamma_x
        b_local_examples[key] = b_local_from_gamma(x, gamma_x, S)

b_vals = np.array(b_vals, dtype=float)
mse_vals = np.array(mse_vals, dtype=float)

print("b range:", float(np.min(b_vals)), "to", float(np.max(b_vals)))
print("fit MSE range:", float(np.min(mse_vals)), "to", float(np.max(mse_vals)))


## Plot 1 — Universality exponent across blockade boundary

In [ ]:
plt.figure(figsize=(8.2, 5.0))
plt.plot(ratio_vals, b_vals, marker="o")
plt.axvline(1.0, linestyle="--", linewidth=1.5, label="blockade onset: V/Ω ≈ 1")
plt.xlabel("V / Ω")
plt.ylabel("fitted global exponent b")
plt.title("Universality transition across the blockade boundary")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 2 — Sensitivity of b to interaction strength

In [ ]:
db_dV = np.gradient(b_vals, ratio_vals)

plt.figure(figsize=(8.2, 5.0))
plt.plot(ratio_vals, db_dV, marker="o")
plt.axvline(1.0, linestyle="--", linewidth=1.5, label="blockade onset")
plt.xlabel("V / Ω")
plt.ylabel("db / d(V/Ω)")
plt.title("Sensitivity of the universality exponent near blockade")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 3 — Representative decay curves

In [ ]:
plt.figure(figsize=(8.4, 5.1))
for label, S in S_examples.items():
    ratio = float(label.split("=")[1])
    idx = int(np.argmin(np.abs(ratio_vals - ratio)))
    plt.plot(x, S, label=f'{label}, b={b_vals[idx]:.3f}')
plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Decay curves across the blockade transition")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 4 — Effective-rate geometry

In [ ]:
plt.figure(figsize=(8.4, 5.1))
for label, gamma_x in gamma_examples.items():
    plt.plot(x, gamma_x, label=label)
plt.xlabel("x")
plt.ylabel("Γ_eff(x)")
plt.title("Effective-rate deformation across blockade")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 5 — Local exponent fields

In [ ]:
plt.figure(figsize=(8.4, 5.1))
for label, b_local in b_local_examples.items():
    plt.plot(x, b_local, label=label)
plt.xlabel("x")
plt.ylabel("b_local(x)")
plt.title("Local exponent field across blockade")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 6 — Fit quality across blockade ratio

In [ ]:
plt.figure(figsize=(8.0, 4.9))
plt.plot(ratio_vals, np.log10(mse_vals + 1e-16), marker="o")
plt.axvline(1.0, linestyle="--", linewidth=1.5, label="blockade onset")
plt.xlabel("V / Ω")
plt.ylabel("log10 fit MSE")
plt.title("Fit quality across blockade ratio")
plt.legend()
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Blockade boundary and universality transition")
lines.append("")
lines.append(f"b range across V/Ω sweep: {float(np.min(b_vals)):.6f} to {float(np.max(b_vals)):.6f}")
lines.append(f"Minimum db/d(V/Ω): {float(np.min(db_dV)):.6f}")
lines.append(f"Maximum db/d(V/Ω): {float(np.max(db_dV)):.6f}")
lines.append("")
lines.append("Interpretation:")
lines.append("- As V/Ω increases, the effective-rate process becomes more structured.")
lines.append("- The fitted exponent b shifts across the blockade boundary.")
lines.append("- The local exponent field b_local(x) deforms strongly near and above V/Ω ≈ 1.")
lines.append("- This supports interpreting b as a detector of many-body constraint onset.")
print("\n".join(lines))


## Final conclusion

This notebook connects the universality framework directly to the simplified
Rydberg blockade condition.

We find that:

- the exponent `b` changes systematically with `V / Ω`,
- the strongest deformation occurs near the blockade boundary,
- the local exponent field `b_local(x)` records the onset of interaction-dominated dynamics.

So the final physical interpretation is:

`V / Ω`  
→ blockade onset  
→ deformation of `Γ_eff(x)`  
→ deformation of `b_local(x)`  
→ shift in global exponent `b`
